# Generative AI — Ask the ICU Data

## Objective

This prototype explores the use of Generative AI as an interpretation layer for ICU analytics.

Instead of allowing the language model to directly query the clinical database, validated metrics are calculated in Python and provided to the LLM as structured analytical context.

The goal is to generate concise executive explanations while reducing the risk of hallucinated numerical results.

> This prototype is intended for exploratory analytics and portfolio demonstration purposes. It is not a clinical decision-support system.

In [75]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/icu_analytics_v2.csv"
)

print("ICU stays:", df["patientunitstayid"].nunique())

df.head()

ICU stays: 2520


,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,hospitaladmitsource,hospitaldischargestatus,...,n_medication_records,n_lab_records,apache_score,predicted_icu_mortality,predicted_icu_los,actual_vent_days,apache_version,ventilation_status,ventilation_data_available,age_numeric
0,141764,129391,Female,87,Caucasian,59,91,NaN,NaN,Alive,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,NaN,87.0
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",NaN,Alive,...,14.0,72.0,47.0,0.012311,1.374807,NaN,IVa,Unknown,0.0,87.0
2,143870,131022,Male,76,Caucasian,68,103,"Endarterectomy, carotid",Operating Room,Alive,...,40.0,49.0,60.0,0.015707,3.021671,NaN,IVa,Unknown,0.0,76.0
3,144815,131736,Female,34,Caucasian,56,82,"Overdose, other toxin, poison or drug",Emergency Department,Alive,...,5.0,74.0,25.0,0.002133,0.806311,NaN,IVa,Unknown,0.0,34.0
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",Emergency Department,Alive,...,40.0,94.0,37.0,0.007556,3.503540,NaN,IVa,Unknown,0.0,61.0


## Analytics Engine

The functions below calculate validated ICU metrics directly from the analytical dataset. These results will later be provided to the language model as structured context.

In [76]:
def calculate_icu_metrics(data):
    return {
        "icu_stays": int(data["patientunitstayid"].nunique()),
        "average_los_days": round(data["los_days"].mean(), 2),
        "median_los_days": round(data["los_days"].median(), 2),
        "icu_mortality_rate": round(data["icu_death"].mean() * 100, 2),
        "average_apache_score": round(data["apache_score"].mean(), 2)
    }


icu_metrics = calculate_icu_metrics(df)

icu_metrics

{'icu_stays': 2520,
 'average_los_days': np.float64(2.42),
 'median_los_days': np.float64(1.47),
 'icu_mortality_rate': np.float64(5.0),
 'average_apache_score': np.float64(53.16)}

In [77]:
{
 'icu_stays': 2520,
 'average_los_days': 2.42,
 'median_los_days': ...,
 'icu_mortality_rate': 5.0,
 'average_apache_score': ...
}

{'icu_stays': 2520,
 'average_los_days': 2.42,
 'median_los_days': Ellipsis,
 'icu_mortality_rate': 5.0,
 'average_apache_score': Ellipsis}

In [78]:
def analyze_by_unit(data):
    summary = (
        data.groupby("unittype")
        .agg(
            icu_stays=("patientunitstayid", "nunique"),
            average_los=("los_days", "mean"),
            mortality_rate=("icu_death", "mean"),
            average_apache=("apache_score", "mean")
        )
        .reset_index()
    )

    summary["mortality_rate"] = (
        summary["mortality_rate"] * 100
    )

    return summary.round(2)


unit_analysis = analyze_by_unit(df)

unit_analysis

,unittype,icu_stays,average_los,mortality_rate,average_apache
0,CCU-CTICU,82,2.71,7.32,50.70
1,CSICU,65,1.81,3.08,60.96
2,CTICU,52,3.07,3.85,51.11
3,Cardiac ICU,133,2.48,4.51,50.73
4,MICU,142,2.32,4.93,60.48
5,Med-Surg ICU,1898,2.40,5.01,52.97
6,Neuro ICU,65,2.31,6.15,48.59
7,SICU,83,2.86,4.82,54.69


In [79]:
def analyze_ventilation(data):
    summary = (
        data.groupby(
            "ventilation_status",
            dropna=False
        )
        .agg(
            icu_stays=("patientunitstayid", "nunique"),
            average_los=("los_days", "mean"),
            mortality_rate=("icu_death", "mean"),
            average_apache=("apache_score", "mean")
        )
        .reset_index()
    )

    summary["mortality_rate"] = (
        summary["mortality_rate"] * 100
    )

    return summary.round(2)


ventilation_analysis = analyze_ventilation(df)

ventilation_analysis

,ventilation_status,icu_stays,average_los,mortality_rate,average_apache
0,Unknown,1999,1.97,3.45,47.21
1,Ventilated,521,4.14,10.94,68.24


In [80]:
analytics_context = f"""
ICU ANALYTICS SUMMARY

Overall metrics:
{icu_metrics}

Metrics by ICU type:
{unit_analysis.to_string(index=False)}

Metrics by ventilation data status:
{ventilation_analysis.to_string(index=False)}

Important data limitation:
The category 'Unknown' in ventilation_status indicates unavailable
or insufficient ventilation information. It must NOT be interpreted
as absence of mechanical ventilation.
"""

print(analytics_context)


ICU ANALYTICS SUMMARY

Overall metrics:
{'icu_stays': 2520, 'average_los_days': np.float64(2.42), 'median_los_days': np.float64(1.47), 'icu_mortality_rate': np.float64(5.0), 'average_apache_score': np.float64(53.16)}

Metrics by ICU type:
    unittype  icu_stays  average_los  mortality_rate  average_apache
   CCU-CTICU         82         2.71            7.32           50.70
       CSICU         65         1.81            3.08           60.96
       CTICU         52         3.07            3.85           51.11
 Cardiac ICU        133         2.48            4.51           50.73
        MICU        142         2.32            4.93           60.48
Med-Surg ICU       1898         2.40            5.01           52.97
   Neuro ICU         65         2.31            6.15           48.59
        SICU         83         2.86            4.82           54.69

Metrics by ventilation data status:
ventilation_status  icu_stays  average_los  mortality_rate  average_apache
           Unknown       19

In [81]:
question = """
What are the main factors associated with differences
in ICU length of stay in this dataset?
"""

In [82]:
prompt = f"""
You are an analytics assistant supporting ICU performance analysis.

Use ONLY the numerical information provided in the analytical context.

Do not invent numbers.
Do not infer causality from associations.
Do not provide medical advice.
Clearly distinguish observed findings from possible interpretations.
Mention relevant data limitations when necessary.

ANALYTICAL CONTEXT:

{analytics_context}

USER QUESTION:

{question}

Provide a concise executive analysis using the following structure:

1. Key finding
2. Supporting evidence
3. Possible interpretation
4. Data limitations
"""

print(prompt)


You are an analytics assistant supporting ICU performance analysis.

Use ONLY the numerical information provided in the analytical context.

Do not invent numbers.
Do not infer causality from associations.
Do not provide medical advice.
Clearly distinguish observed findings from possible interpretations.
Mention relevant data limitations when necessary.

ANALYTICAL CONTEXT:


ICU ANALYTICS SUMMARY

Overall metrics:
{'icu_stays': 2520, 'average_los_days': np.float64(2.42), 'median_los_days': np.float64(1.47), 'icu_mortality_rate': np.float64(5.0), 'average_apache_score': np.float64(53.16)}

Metrics by ICU type:
    unittype  icu_stays  average_los  mortality_rate  average_apache
   CCU-CTICU         82         2.71            7.32           50.70
       CSICU         65         1.81            3.08           60.96
       CTICU         52         3.07            3.85           51.11
 Cardiac ICU        133         2.48            4.51           50.73
        MICU        142         2.32

In [83]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key loaded successfully.


In [84]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key loaded successfully.


## LLM Integration

The validated analytical context is now provided to a language model for executive interpretation.

The LLM does not calculate the underlying ICU metrics. Its role is limited to interpreting results previously calculated and validated in Python.

In [85]:
import json
import urllib.request

url = "http://localhost:11434/api/chat"

payload = {
    "model": "qwen2.5:3b",
    "messages": [
        {
            "role": "user",
            "content": "Reply only with: Ollama connection successful."
        }
    ],
    "stream": False
}

request = urllib.request.Request(
    url,
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(request) as response:
    result = json.loads(response.read().decode("utf-8"))

print(result["message"]["content"])

Ollama connection successful.


In [86]:
payload = {
    "model": "qwen2.5:3b",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "stream": False
}

request = urllib.request.Request(
    url,
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(request) as response:
    result = json.loads(response.read().decode("utf-8"))

print(result["message"]["content"])

### Executive Analysis

#### Key Finding
The main factors associated with differences in ICU length of stay (LOS) in this dataset are ICU type and the presence of mechanical ventilation. The LOS for ICUs with ventilation status as "Unknown" is significantly shorter compared to those with mechanical ventilation.

#### Supporting Evidence
- **ICU Type**: ICUs of different types (CCU-CTICU, CSICU, CTICU, Cardiac ICU, MICU, Med-Surg ICU, Neuro ICU, SICU) exhibit varying LOS. For instance, ICUs with a "Ventilated" ventilation status have an average LOS of 4.14 days, while those in the "Unknown" category have an average LOS of 1.97 days.
- **Ventilation Status**: The "Ventilated" category, which includes patients with available ventilation status data, has an average LOS of 4.14 days compared to 1.97 days for the "Unknown" category.

#### Possible Interpretation
The significant difference in LOS between the "Ventilated" and "Unknown" categories suggests that the presence of available ventila

In [87]:
def ask_icu_data(question):
    local_prompt = f"""
You are an analytics assistant supporting ICU performance analysis.

Use ONLY the numerical information provided in the analytical context.

Rules:
- Do not invent numbers.
- Do not infer causality from associations.
- Do not provide medical advice.
- Clearly distinguish observed findings from possible interpretations.
- Mention relevant data limitations.
- 'Unknown' ventilation status means unavailable or insufficient information
  and must NOT be interpreted as absence of mechanical ventilation.

ANALYTICAL CONTEXT:

{analytics_context}

USER QUESTION:

{question}

Provide a concise executive analysis using:

1. Key finding
2. Supporting evidence
3. Possible interpretation
4. Data limitations
"""

    payload = {
        "model": "qwen2.5:3b",
        "messages": [
            {
                "role": "user",
                "content": local_prompt
            }
        ],
        "stream": False
    }

    request = urllib.request.Request(
        "http://localhost:11434/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(request) as response:
        result = json.loads(
            response.read().decode("utf-8")
        )

    return result["message"]["content"]

In [88]:
question = """
Which ICU units have the highest average length of stay,
and what patterns can be observed in the available data?
"""

print(
    ask_icu_data(question)
)

### Executive Analysis

1. **Key Finding:**
   - The ICU unit with the highest average length of stay (LOS) is **Med-Surg ICU**, with an average LOS of 2.40 days.

2. **Supporting Evidence:**
   - The data shows that the Med-Surg ICU has the highest number of ICU stays among all ICU types, totaling 1898 stays. This is significantly higher than the other ICU types.
   - The average LOS for the Med-Surg ICU is 2.40 days, which is the highest value among all ICU types.

3. **Possible Interpretation:**
   - The Med-Surg ICU likely handles a wide range of medical conditions with varying severity levels, which could explain the longer average LOS. These patients may require longer hospital stays due to their underlying conditions, treatment complexity, or recovery periods.
   - Additionally, the higher mortality rate and average APACHE score for the Med-Surg ICU suggest that these patients may be more critically ill, requiring more intensive care and monitoring.

4. **Data Limitations:**
   

## Conclusions

This prototype demonstrates a local Generative AI layer integrated with ICU analytics.

All clinical metrics are calculated deterministically in Python before being provided to the language model. The LLM is used only to interpret validated analytical results in natural language.

The implementation uses a local LLM through the Ollama HTTP API, avoiding external transmission of the analytical dataset and reducing dependency on paid APIs.

Prompt-level guardrails were implemented to:
- prevent unsupported numerical claims;
- avoid causal interpretations;
- avoid medical advice;
- preserve explicit data-quality limitations.

This prototype is intended for analytics and portfolio demonstration purposes and is not a clinical decision-support system.